In [1]:
import torch
from torch import nn
import matplotlib.pyplot as plt
import re
import collections
import torch.nn.functional as F

# Text dataset

In [2]:
def read_data():
    with open("timemachine.txt", 'r') as f:
        return f.read()
    
def process_text(text):
    return re.sub("[^A-Za-z]+", ' ', text).lower()

def tokenize(text): return list(text)

In [13]:
class Vocab:
    def __init__(self, tokens, min_req=0, reserved_tokens=[]):
        if isinstance(tokens[0], list):
            tokens = [token for line in tokens for token in line]
        counter = collections.Counter(tokens)
        self.freq_tokens = sorted(counter.items(), key=lambda x: x[1], reverse=True)
        self.idx_to_token = list(sorted(set(["<unk>"] + reserved_tokens + [
            t for t,f in self.freq_tokens if f>=min_req
        ])))
        self.token_to_idx = {t:i for i,t in enumerate(self.idx_to_token)}

    def __len__(self): return len(self.idx_to_token)

    def __getitem__(self, tokens):
        if not isinstance(tokens, (tuple, list)):
            return self.token_to_idx[tokens]
        return [self.__getitem__(t) for t in tokens]

    def to_tokens(self, indices):
        if hasattr(indices, "__len__") and len(indices):
            return [self.idx_to_token[i] for i in indices]
        return self.idx_to_token[indices]

    @property
    def unk(self): return self.token_to_idx["<unk>"]

vocab = Vocab(tokenize(process_text(read_data())))
len(vocab)

28

In [30]:
class TextData(torch.utils.data.Dataset):
    def __init__(self, text, num_steps):
        tokens = tokenize(text)
        self.vocab = Vocab(tokens)
        self.vocab_size = len(vocab)
        self.corpus = [self.vocab[t] for t in tokens]
        self.array = torch.tensor([self.corpus[i:i+num_steps+1] for i in range(len(self.corpus)-num_steps)])
        self.X = self.array[:,:-1]
        self.y = self.array[:,1:]

text_data = TextData(process_text(read_data()), 32)

In [23]:
text_data.array

tensor([[21,  9,  6,  ..., 10,  0, 21],
        [ 9,  6,  0,  ...,  0, 21,  9],
        [ 6,  0, 21,  ..., 21,  9,  6],
        ...,
        [20, 21, 10,  ...,  0, 14,  2],
        [21, 10, 13,  ..., 14,  2, 15],
        [10, 13, 13,  ...,  2, 15,  0]])

In [31]:
text_data.corpus

[21,
 9,
 6,
 0,
 21,
 10,
 14,
 6,
 0,
 14,
 2,
 4,
 9,
 10,
 15,
 6,
 0,
 3,
 26,
 0,
 9,
 0,
 8,
 0,
 24,
 6,
 13,
 13,
 20,
 0,
 10,
 0,
 21,
 9,
 6,
 0,
 21,
 10,
 14,
 6,
 0,
 21,
 19,
 2,
 23,
 6,
 13,
 13,
 6,
 19,
 0,
 7,
 16,
 19,
 0,
 20,
 16,
 0,
 10,
 21,
 0,
 24,
 10,
 13,
 13,
 0,
 3,
 6,
 0,
 4,
 16,
 15,
 23,
 6,
 15,
 10,
 6,
 15,
 21,
 0,
 21,
 16,
 0,
 20,
 17,
 6,
 2,
 12,
 0,
 16,
 7,
 0,
 9,
 10,
 14,
 0,
 24,
 2,
 20,
 0,
 6,
 25,
 17,
 16,
 22,
 15,
 5,
 10,
 15,
 8,
 0,
 2,
 0,
 19,
 6,
 4,
 16,
 15,
 5,
 10,
 21,
 6,
 0,
 14,
 2,
 21,
 21,
 6,
 19,
 0,
 21,
 16,
 0,
 22,
 20,
 0,
 9,
 10,
 20,
 0,
 8,
 19,
 6,
 26,
 0,
 6,
 26,
 6,
 20,
 0,
 20,
 9,
 16,
 15,
 6,
 0,
 2,
 15,
 5,
 0,
 21,
 24,
 10,
 15,
 12,
 13,
 6,
 5,
 0,
 2,
 15,
 5,
 0,
 9,
 10,
 20,
 0,
 22,
 20,
 22,
 2,
 13,
 13,
 26,
 0,
 17,
 2,
 13,
 6,
 0,
 7,
 2,
 4,
 6,
 0,
 24,
 2,
 20,
 0,
 7,
 13,
 22,
 20,
 9,
 6,
 5,
 0,
 2,
 15,
 5,
 0,
 2,
 15,
 10,
 14,
 2,
 21,
 6,
 5,
 0,
 21,
 9,
 6,


In [28]:
num_train = 10000
num_val = 5000
train_indices = slice(0, num_train)
eval_indices = slice(num_train, num_train+num_val)
batch_size = 1024

def get_dataloader(text_data, indices, batch_size, shuffle):
    data = torch.utils.data.TensorDataset(text_data.X[indices], text_data.y[indices])
    return torch.utils.data.DataLoader(data, batch_size=batch_size, shuffle=shuffle)

train_dataloader = get_dataloader(text_data, train_indices, batch_size, True)
eval_dataloader = get_dataloader(text_data, eval_indices, batch_size, False)

In [ ]:
def grad_clip(clip_val, model):
    params = [p for p in model.parameters() if p.requires_grad]
    norm = torch.sqrt(sum(torch.sum((p.grad**2)) for p in params))
    if norm > clip_val:
        for p in params:
            p.grad[:] *= clip_val / norm

In [ ]:
def train_lstm(device, model, max_epochs, clip_val, lr, train_dataloader, eval_dataloader, report_every):
    metrics = {"train_loss": [], "train_ppl": [], "eval_loss": [], "eval_ppl": []}
    criterion = nn.CrossEntropyLoss()

    model = model.to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr)

    for epoch in range(max_epochs):
        model.train()
        num_instances = 0
        epoch_loss = 0.
        epoch_ppl = 0.
        for step, batch in enumerate(train_dataloader):
            optimizer.zero_grad()
            batch = [a.to(device) for a in batch]
            bs = batch[-1].size(0)
            logits = model(*batch[:-1])
            logits = logits.reshape(-1, logits.shape[-1])
            loss = criterion(logits, batch[-1].reshape(-1))
            loss.backward()
            grad_clip(clip_val, model)
            optimizer.step()
            epoch_loss += loss.item() * bs
            epoch_ppl += torch.exp(loss).item() * bs
            num_instances += bs

        epoch_loss /= num_instances
        epoch_ppl /= num_instances
        metrics["train_loss"].append(epoch_loss)
        metrics["train_ppl"].append(epoch_ppl)
        if epoch % report_every==0:
            print(f"[{epoch}/{max_epochs}] train_loss: {epoch_loss:.5f}, train_ppl: {epoch_ppl:.5f}")

        model.eval()
        num_instances = 0
        epoch_loss = 0.
        epoch_ppl = 0.
        for step, batch in enumerate(eval_dataloader):
            batch = [a.to(device) for a in batch]
            bs = batch[-1].size(0)
            with torch.no_grad():
                logits = model(*batch[:-1])
                logits = logits.reshape(-1, logits.shape[-1])
                loss = criterion(logits, batch[-1].reshape(-1))
                epoch_loss += loss.item() * bs
                epoch_ppl += torch.exp(loss).item() * bs
            num_instances += bs

        epoch_loss /= num_instances
        epoch_ppl /= num_instances
        metrics["eval_loss"].append(epoch_loss)
        metrics["eval_ppl"].append(epoch_ppl)
        if epoch % report_every==0:
            print(f"[{epoch}/{max_epochs}] eval_loss: {epoch_loss:.5f}, eval_ppl: {epoch_ppl:.5f}")
            
    return metrics
    

In [ ]:
len(text_data.vocab)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vocab_size = text_data.vocab_size
rnn = RNNfromScratch(num_inputs=vocab_size, num_hiddens=32)
model = RNNLMfromScratch(rnn=rnn, vocab_size=vocab_size)
metrics = train_rnn(model, max_epochs=100, clip_val=1, lr=1, train_dataloader=train_dataloader, eval_dataloader=eval_dataloader, report_every=10)


In [ ]:
plt.plot(metrics["train_ppl"], label="train")
plt.plot(metrics["eval_ppl"], label="eval")
plt.legend(); plt.show()

In [ ]:
def predict_last(prefix, num_preds, model, vocab, device):
    state, outputs = None, [vocab[prefix[0]]]
    model.eval()
    with torch.no_grad():
        for i in range(len(prefix)+num_preds-1):
            if i<len(prefix)-1:
                outputs.append(vocab[prefix[i+1]])
            else:
                x = torch.tensor([[outputs[-1]]], device=device)
                logits = model(x, state)
                pred_idx = logits.argmax(dim=2).reshape(1)
                outputs.append(int(pred_idx))
        return "".join(vocab.idx_to_token[i] for i in outputs)

def predict(prefix, num_preds, model, vocab, device):
    state, outputs = None, [vocab[prefix[0]]]
    model.eval()
    with torch.no_grad():
        for i in range(len(prefix)+num_preds-1):
            # x = torch.tensor([[outputs[-1]]], device=device)
            if i<len(prefix)-1:
                outputs.append(vocab[prefix[i+1]])
            else:
                x = torch.tensor([outputs], device=device)
                logits = model(x, state)
                pred_idx = logits.argmax(dim=2).reshape(-1)
                outputs.append(int(pred_idx[-1]))
        return "".join(vocab.idx_to_token[i] for i in outputs)



In [ ]:
predict_last("it has", 20, model, text_data.vocab, device)

In [ ]:
predict("it has", 20, model, text_data.vocab, device)

In [ ]:
vocab_size = text_data.vocab_size
rnn = RNN(num_inputs=vocab_size, num_hiddens=32)
model = RNNLM(rnn=rnn, vocab_size=vocab_size)
metrics = train_rnn(model, max_epochs=100, clip_val=1, lr=1, train_dataloader=train_dataloader, eval_dataloader=eval_dataloader, report_every=10)


In [ ]:
plt.plot(metrics["train_ppl"], label="train")
plt.plot(metrics["eval_ppl"], label="eval")
plt.legend(); plt.show()

In [ ]:
predict_last("it has", 20, model, text_data.vocab, device)

In [ ]:
predict("it has", 20, model, text_data.vocab, device)